In [9]:
import pandas as pd
import numpy as np
from sodapy import Socrata

##ingest sample data - 50k rows from checkoutyear 2015

# Connect to Seattle Open Data (no auth needed for public data)
client = Socrata("data.seattle.gov", None)

# Fetch records where CheckoutYear is 2015
# limit= controls how many rows you get (dataset has millions of rows!)
results = client.get("tmmm-ytt6",
                     where="checkoutyear=2015",
                     limit=50000)

# Convert to a pandas DataFrame
df = pd.DataFrame.from_records(results)
print(f"Rows loaded: {len(df)}")
df.head()

Rows loaded: 50000


,usageclass,checkouttype,materialtype,checkoutyear,checkoutmonth,checkouts,title,subjects,creator,publisher,publicationyear
0,Physical,Horizon,BOOK,2015,1,1,Paradise walk,"Women historians Fiction, Mystery fiction",NaN,NaN,NaN
1,Digital,Freegal,SONG,2015,1,1,Gates of Eden,NaN,Arlo Guthrie,NaN,NaN
2,Physical,Horizon,BOOK,2015,1,1,martial arts book,Martial arts Juvenile literature,NaN,NaN,NaN
3,Digital,Freegal,SONG,2015,1,2,"Side 3, Pt. 7: Talkin' Hawkin'",NaN,Pink Floyd,NaN,NaN
4,Digital,Hoopla,MUSIC,2015,1,1,Jerry Christmas,Holiday,NaN,eOne Music,NaN


In [ ]:
##summary statistics - count of titles checked out by materialtype and checkouttype

# Filter rows
filtered_df = df[
    (df['usageclass'].isin(['Physical', 'Digital']))
]

# Group and aggregate
summary = filtered_df.groupby(['materialtype', 'checkouttype'])['title'].nunique().reset_index()
summary.columns = ['materialtype', 'checkouttype', 'num_titles']

summary.sort_values('num_titles', ascending = False)

## Ingest all checkout data, chunked out by year

In [3]:
# imports
import os
import glob
import pandas as pd
from sodapy import Socrata
from dotenv import load_dotenv
from tqdm import tqdm
import time

# load token
load_dotenv('/Users/audriswong/data-portfolio/projects/seattle-checkouts/.env')

# helper functions
def get_completed_offsets(checkpoint_dir):
    files = glob.glob(f'{checkpoint_dir}/batch_*.parquet')
    completed = set()
    for f in files:
        try:
            offset = int(os.path.basename(f)
                        .replace('batch_', '').replace('.parquet', ''))
            completed.add(offset)
        except:
            pass
    return completed

def fetch_with_retry(client, dataset_id, where, limit, offset, max_retries=5):
    wait = 2
    for attempt in range(max_retries):
        try:
            batch = client.get(dataset_id,
                               where=where,
                               limit=limit,
                               offset=offset)
            return batch
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"\n❌ Failed after {max_retries} attempts at offset {offset}: {e}")
                raise
            print(f"\n⚠️  Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
            wait *= 10

def ingest_with_checkpoint(checkpoint_dir='/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/raw/checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    client = Socrata("data.seattle.gov",
                     os.getenv("SEATTLE_API_TOKEN"),
                     timeout=180)
    
    total_rows = 22865320
    rows_already_done = 18010000

    completed_offsets = get_completed_offsets(checkpoint_dir)
    print(f"🔄 Found {len(completed_offsets)} saved batches — resuming...")
    print(f"   Rows already saved: {rows_already_done:,} of {total_rows:,}")

    batch_size = 10000
    offset = 17500000

    with tqdm(total=total_rows, initial=rows_already_done, unit="rows") as progress_bar:
        while True:
            if offset in completed_offsets:
                progress_bar.update(batch_size)
                offset += batch_size
                continue

            batch = fetch_with_retry(client,
                                     dataset_id="tmmm-ytt6",
                                     where="checkoutyear >= 2017",
                                     limit=batch_size,
                                     offset=offset)
            if not batch:
                break

            batch_df = pd.DataFrame.from_records(batch)
            batch_df.to_parquet(f'{checkpoint_dir}/batch_{offset}.parquet')
            progress_bar.update(len(batch))
            offset += batch_size
            time.sleep(3)

    print("\n🔧 Combining all batches...")
    all_files = sorted(glob.glob(f'{checkpoint_dir}/batch_*.parquet'),
                       key=lambda x: int(os.path.basename(x)
                       .replace('batch_', '').replace('.parquet', '')))
    df_all = pd.concat([pd.read_parquet(f) for f in all_files], ignore_index=True)
    print(f"✅ Done! Total rows loaded: {len(df_all):,}")
    return df_all

# run it
#df_all = ingest_with_checkpoint()

## Running into high offset timeouts when ingesting this way - switching to year-based ingestion, and checking to see where the offset methodology left off

In [18]:
#find gap from switching from batch size 50k to 10k

import glob
import os
import pandas as pd

files = sorted(glob.glob('checkpoints/batch_*.parquet'),
               key=lambda x: int(os.path.basename(x)
               .replace('batch_', '').replace('.parquet', '')))

# Build a map of offset → row count for saved batches
saved_batches = {}
for f in files:
    offset = int(os.path.basename(f).replace('batch_', '').replace('.parquet', ''))
    rows = len(pd.read_parquet(f))
    saved_batches[offset] = rows

# Walk through sequence to find true gaps
covered_up_to = 0
gaps = []

for offset in sorted(saved_batches.keys()):
    if offset > covered_up_to:
        # There's a gap between covered_up_to and this offset
        gaps.append((covered_up_to, offset))
    covered_up_to = offset + saved_batches[offset]

print(f"Rows covered:  {covered_up_to:,}")
print(f"True gaps:     {len(gaps)}")
if gaps:
    for start, end in gaps[:10]:
        print(f"  Missing rows {start:,} → {end:,} ({end-start:,} rows)")

#fill the gap

from sodapy import Socrata
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
client = Socrata("data.seattle.gov",
                 os.getenv("SEATTLE_API_TOKEN"),
                 timeout=180)

# Fetch just the missing 10,000 rows
print("Fetching missing rows 640,000 → 650,000...")
batch = fetch_with_retry(client,
                         dataset_id="tmmm-ytt6",
                         os.getenv("SEATTLE_API_TOKEN"),
                         where="checkoutyear >= 2017",
                         limit=10000,
                         offset=640000)

# Save as a checkpoint file
pd.DataFrame.from_records(batch).to_parquet('checkpoints/batch_640000.parquet')
print(f"✅ Saved {len(batch):,} rows")

Rows covered:  17,500,000
True gaps:     1
  Missing rows 640,000 → 650,000 (10,000 rows)
Fetching missing rows 640,000 → 650,000...
✅ Saved 10,000 rows


In [3]:
#check which years are already covered

import pandas as pd
import glob
import random

checkpoint_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/raw/checkpoints'

# Sample 10 random batch files to see year distribution
files = glob.glob(f'{checkpoint_dir}/batch_*.parquet')
sample_files = random.sample(files, 10)

years = []
for f in sample_files:
    df = pd.read_parquet(f)
    years.extend(df['checkoutyear'].unique().tolist())

print(sorted(set(years)))

['2017', '2018', '2020', '2021', '2022']


In [4]:
# Check what year the last saved batch contains
last_batch = pd.read_parquet(f'{checkpoint_dir}/batch_17450000.parquet')
print(last_batch['checkoutyear'].value_counts())

checkoutyear
2024    50000
Name: count, dtype: int64


In [4]:
#check year coverage
import pandas as pd
import glob

checkpoint_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/raw/checkpoints'

files = glob.glob(f'{checkpoint_dir}/batch_*.parquet')
year_counts = {}

for f in files:
    df = pd.read_parquet(f)
    for year, count in df['checkoutyear'].value_counts().items():
        year_counts[year] = year_counts.get(year, 0) + count

for year in sorted(year_counts.keys()):
    print(f"{year}: {year_counts[year]:,} rows")

2017: 3,340,433 rows
2018: 2,665,098 rows
2019: 2,589,001 rows
2020: 1,721,376 rows
2021: 2,242,272 rows
2022: 2,531,508 rows
2023: 2,676,026 rows
2024: 254,286 rows


In [1]:
#check to see how many rows of data exist within the API

from sodapy import Socrata
from dotenv import load_dotenv
import os

load_dotenv('/Users/audriswong/data-portfolio/projects/seattle-checkouts/.env')
client = Socrata("data.seattle.gov",
                 os.getenv("SEATTLE_API_TOKEN"),
                 timeout=180)

for year in [2024, 2025]:
    result = client.get("tmmm-ytt6",
                        select="count(*)",
                        where=f"checkoutyear={year}")
    print(f"{year}: {int(result[0]['count']):,} total rows in API")

/Users/audriswong/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


2024: 2,558,970 total rows in API
2025: 2,814,717 total rows in API


## Ingest Remaining years - 2024, 2025 and 2026

In [7]:
def ingest_remaining_by_year(checkpoint_dir='/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/raw/checkpoints'):
    
    client = Socrata("data.seattle.gov",
                     os.getenv("SEATTLE_API_TOKEN"),
                     timeout=180)

    # We have 18,010,000 rows already saved covering 2017 through most of the dataset
    # Fetch remaining data year by year to avoid high offset timeouts
    # Based on 79% complete, we're likely missing late 2023 through 2025
    years_to_check = [2025]
    #[2024, 2025, 2026]  # 2024 partial, 2025-2026 missing entirely

    completed_offsets = get_completed_offsets(checkpoint_dir)

    for year in years_to_check:
        print(f"\n📅 Checking year {year}...")
        
        # Get total for this year
        count_result = client.get("tmmm-ytt6",
                                   select="count(*)",
                                   where=f"checkoutyear={year}")
        year_total = int(count_result[0]['count'])
        print(f"   {year_total:,} total rows for {year}")

        offset = 0
        batch_size = 10000

        with tqdm(total=year_total, unit="rows", desc=str(year)) as progress_bar:
            while True:
                # Use year-specific checkpoint naming to avoid conflicts
                checkpoint_file = f'{checkpoint_dir}/year_{year}_batch_{offset}.parquet'
                
                if os.path.exists(checkpoint_file):
                    progress_bar.update(batch_size)
                    offset += batch_size
                    continue

                batch = fetch_with_retry(client,
                                         dataset_id="tmmm-ytt6",
                                         where=f"checkoutyear={year}",
                                         limit=batch_size,
                                         offset=offset)
                if not batch:
                    break

                pd.DataFrame.from_records(batch).to_parquet(checkpoint_file)
                progress_bar.update(len(batch))
                offset += batch_size
                time.sleep(5)

        print(f"✅ Year {year} complete")

# run it
ingest_remaining_by_year()


📅 Checking year 2025...
   2,814,717 total rows for 2025


2025:  34%|█████████████████████▍                                         | 960000/2814717 [01:05<01:32, 20133.75rows/s]


⚠️  Attempt 1 failed: HTTPSConnectionPool(host='data.seattle.gov', port=443): Read timed out. (read timeout=180). Retrying in 2s...

⚠️  Attempt 2 failed: HTTPSConnectionPool(host='data.seattle.gov', port=443): Read timed out. (read timeout=180). Retrying in 20s...

⚠️  Attempt 3 failed: HTTPSConnectionPool(host='data.seattle.gov', port=443): Read timed out. (read timeout=180). Retrying in 200s...

⚠️  Attempt 4 failed: HTTPSConnectionPool(host='data.seattle.gov', port=443): Read timed out. (read timeout=180). Retrying in 2000s...


2025:  34%|█████████████████████▍                                         | 960000/2814717 [32:24<1:02:35, 493.80rows/s]


KeyboardInterrupt: 

In [10]:
#combine batches
checkpoint_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/raw/checkpoints'

all_files = glob.glob(f'{checkpoint_dir}/batch_*.parquet') + \
            glob.glob(f'{checkpoint_dir}/year_*_batch_*.parquet')

print(f"Total files to combine: {len(all_files)}")

df_all = pd.concat([pd.read_parquet(f) for f in all_files], ignore_index=True)
print(f"Rows before dedup: {len(df_all):,}")

df_all = df_all.drop_duplicates()
print(f"Rows after dedup:  {len(df_all):,}")
print(f"Expected total:    ~22,865,320")

Total files to combine: 940
Rows before dedup: 23,393,687
Rows after dedup:  21,641,346
Expected total:    ~22,865,320


## for the sake of this analysis, I'm interested in tracking checkouts per title, not per title edition (e.g. "Ramona the Brave" 1st edition was checked out once, 3rd edition checked out 3x, interested in tracking 4 checkouts across all editions.  Will keep data granular to edition, but analysis will be collapsed on unique titles.